In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping


2025-02-24 14:20:47.296000: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-24 14:20:47.559903: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-24 14:20:47.776464: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-24 14:20:47.957180: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-24 14:20:48.019849: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-24 14:20:48.374200: I tensorflow/core/platform/cpu_feature_gu

In [2]:
# Carregar os dados
titanic = pd.read_pickle('/home/usp-ds-arnem/data/aula4/titanic1.pkl')
X = titanic.drop(columns='survived')
y = titanic.survived

# Dividir os dados em treino e teste (holdout)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [3]:
# Normalizar os dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# Construir a rede neural com 5 camadas
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Camada oculta 1
    Dense(32, activation='relu'),  # Camada oculta 2
    Dense(16, activation='relu'),  # Camada oculta 3
    Dense(8, activation='relu'),   # Camada oculta 4
    Dense(1, activation='sigmoid') # Camada de saída (classificação binária)
])

/root/.local/share/virtualenvs/usp-ds-arnem-_IUna4d0/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1740417766.133851  211685 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:0b:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-24 14:22:46.285021: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [5]:
# Compilar o modelo
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # Função de perda para classificação binária
    metrics=[AUC(name='auc')]    # Usar AUC como métrica
)

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,329 (13.00 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Treinar o modelo
early_stopping = EarlyStopping(
    monitor='val_auc',  # Monitorar a AUC no conjunto de validação
    patience=10,        # Parar após 10 épocas sem melhoria
    mode='max',         # Maximizar a AUC
    restore_best_weights=True  # Restaurar os melhores pesos encontrados
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,          # Número máximo de épocas
    batch_size=32,       # Tamanho do batch
    callbacks=[early_stopping],  # Usar early stopping
    verbose=1
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - auc: 0.7130 - loss: 0.6962 - val_auc: 0.7935 - val_loss: 0.6338
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.8275 - loss: 0.6053 - val_auc: 0.8350 - val_loss: 0.5646
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.8407 - loss: 0.5334 - val_auc: 0.8457 - val_loss: 0.4891
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - auc: 0.8532 - loss: 0.4699 - val_auc: 0.8555 - val_loss: 0.4409
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.8771 - loss: 0.4226 - val_auc: 0.8617 - val_loss: 0.4355
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.8677 - loss: 0.4055 - val_auc: 0.8646 - val_loss: 0.4273
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.8749 - loss: 0.4053 - val_auc: 0.8670 - val_loss: 0.4238
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.8652 - loss: 0.4296 - val_auc: 0.8669 - val_loss: 0.4296
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.894

In [8]:
# Avaliar o modelo no conjunto de teste
results = model.evaluate(X_test, y_test, verbose=0)
print(f"AUC no teste: {results[1]:.4f}")

AUC no teste: 0.8695
